## KAGGLE SETUP

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set Kaggle path
DATA_PATH = '/kaggle/input/competitions/ieee-fraud-detection/'

# Create output directories
os.makedirs('/kaggle/working/models', exist_ok=True)
os.makedirs('/kaggle/working/outputs', exist_ok=True)
os.makedirs('/kaggle/working/outputs/plots', exist_ok=True)

# Set style for plots
plt.style.use('dark_background')
sns.set_palette('Set2')

print("Kaggle setup complete")
print(f"Data path: {DATA_PATH}")
print(f"Output directories created")

Kaggle setup complete
Data path: /kaggle/input/competitions/ieee-fraud-detection/
Output directories created


## PHASE 1: EXPLORATORY DATA ANALYSIS

In [2]:
print("\n[1.1] Loading data...")
train_tx = pd.read_csv(DATA_PATH + 'train_transaction.csv')
train_id = pd.read_csv(DATA_PATH + 'train_identity.csv')

print(f"Transaction data: {train_tx.shape}")
print(f"Identity data: {train_id.shape}")

# Merge with LEFT join to keep all transactions
train = train_tx.merge(train_id, on='TransactionID', how='left')
print(f"After merge: {train.shape}")
print(f"Memory usage: {train.memory_usage(deep=True).sum() / 1e9:.2f} GB")

# STEP 1.2: Basic Overview
print("\n[1.2] Dataset overview...")
print(f"\nShape: {train.shape}")
print(f"\nData types:\n{train.dtypes.value_counts()}")

# Target distribution
fraud_count = train['isFraud'].sum()
total = len(train)
print(f"\n=== CLASS DISTRIBUTION ===")
print(f"Total transactions: {total:,}")
print(f"Fraudulent: {fraud_count:,} ({fraud_count/total*100:.2f}%)")
print(f"Legitimate: {total-fraud_count:,} ({(total-fraud_count)/total*100:.2f}%)")

# STEP 1.3: Missing Value Analysis
print("\n[1.3] Missing value analysis...")
missing = pd.DataFrame({
    'missing_count': train.isnull().sum(),
    'missing_pct': train.isnull().mean() * 100
}).sort_values('missing_pct', ascending=False)

high_missing = missing[missing['missing_pct'] > 50]
complete = missing[missing['missing_pct'] == 0]

print(f"Features with >50% missing: {len(high_missing)}")
print(f"Complete features (0% missing): {len(complete)}")

# Visualize missing pattern
fig, ax = plt.subplots(figsize=(12, 5))
missing_pct = missing['missing_pct']
bins = [0, 0.1, 10, 30, 50, 70, 100]
labels = ['<0.1%', '0.1-10%', '10-30%', '30-50%', '50-70%', '70-100%']
counts = pd.cut(missing_pct, bins=bins, labels=labels).value_counts().sort_index()
ax.bar(labels, counts.values, color=['#3FB950','#58A6FF','#E3B341','#F78166','#D2A8FF','#FF6B6B'])
ax.set_title('Distribution of Missing Value % Across Features', color='white', fontsize=14)
ax.set_xlabel('Missing %', color='white')
ax.set_ylabel('Number of Features', color='white')
plt.tight_layout()
plt.savefig('/kaggle/working/outputs/plots/01_missing_values_dist.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.close()

print("Missing values plot saved")

# STEP 2: Fraud vs Non-Fraud Analysis
print("\n[2] Fraud vs Non-Fraud Analysis...")

fraud = train[train['isFraud'] == 1]
legit = train[train['isFraud'] == 0]

print("\n=== TRANSACTION AMOUNT STATISTICS ===")
print("\nFraud transactions:")
print(fraud['TransactionAmt'].describe())
print("\nLegit transactions:")
print(legit['TransactionAmt'].describe())

# Plot amount distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw amounts
axes[0].hist(legit['TransactionAmt'].clip(0, 500), bins=50, alpha=0.7,
             label='Legit', color='#3FB950', density=True)
axes[0].hist(fraud['TransactionAmt'].clip(0, 500), bins=50, alpha=0.7,
             label='Fraud', color='#F78166', density=True)
axes[0].set_title('Transaction Amount (clipped at $500)')
axes[0].set_xlabel('Amount ($)')
axes[0].legend()

# Log-scale amounts
axes[1].hist(np.log1p(legit['TransactionAmt']), bins=50, alpha=0.7,
             label='Legit', color='#3FB950', density=True)
axes[1].hist(np.log1p(fraud['TransactionAmt']), bins=50, alpha=0.7,
             label='Fraud', color='#F78166', density=True)
axes[1].set_title('Log(Transaction Amount + 1)')
axes[1].set_xlabel('log1p(Amount)')
axes[1].legend()

for ax in axes:
    ax.set_ylabel('Density')

plt.suptitle('Transaction Amount: Fraud vs Legitimate', fontsize=14)
plt.tight_layout()
plt.savefig('/kaggle/working/outputs/plots/02_amount_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.close()

print("Amount distribution plot saved")

# STEP 3: Time Pattern Analysis
print("\n[3] Time Pattern Analysis...")

# Extract time features
train['hour'] = (train['TransactionDT'] // 3600) % 24
train['day_of_week'] = (train['TransactionDT'] // 86400) % 7

hourly_fraud = train.groupby('hour')['isFraud'].agg(['sum', 'count'])
hourly_fraud['fraud_rate'] = hourly_fraud['sum'] / hourly_fraud['count'] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fraud rate by hour
axes[0].bar(hourly_fraud.index, hourly_fraud['fraud_rate'], color='#F78166', alpha=0.8)
axes[0].axhline(y=train['isFraud'].mean()*100, color='yellow', linestyle='--',
                label=f'Overall rate: {train["isFraud"].mean()*100:.1f}%')
axes[0].set_title('Fraud RATE (%) by Hour of Day')
axes[0].set_xlabel('Hour (0=midnight, 12=noon)')
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].legend()

# Transaction volume by hour
hourly_vol = train.groupby(['hour', 'isFraud']).size().unstack()
hourly_vol.plot(kind='bar', ax=axes[1], color=['#3FB950', '#F78166'], alpha=0.8)
axes[1].set_title('Transaction Volume by Hour')
axes[1].set_xlabel('Hour of Day')
axes[1].legend(['Legit', 'Fraud'])

plt.tight_layout()
plt.savefig('/kaggle/working/outputs/plots/03_time_patterns.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.close()

print("Time pattern plot saved")

# Peak fraud hours
peak_hours = hourly_fraud.nlargest(5, 'fraud_rate')
print("\nTop 5 highest fraud rate hours:")
print(peak_hours[['fraud_rate']])

# STEP 4: Categorical Feature Analysis
print("\n[4] Categorical Feature Analysis...")

# Product code fraud rate
product_fraud = train.groupby('ProductCD')['isFraud'].agg(
    fraud_count='sum',
    total_count='count'
).assign(fraud_rate=lambda x: x.fraud_count / x.total_count * 100)
product_fraud = product_fraud.sort_values('fraud_rate', ascending=False)
print("\nFraud rate by Product Code:")
print(product_fraud)

# Card network fraud rate
card4_fraud = train.groupby('card4')['isFraud'].agg(
    fraud_rate=lambda x: x.mean() * 100,
    total='count'
).sort_values('fraud_rate', ascending=False)
print("\nFraud rate by Card Network:")
print(card4_fraud)

# STEP 5: Class Imbalance Visualization
print("\n[5] Visualizing class imbalance...")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = train['isFraud'].value_counts()
axes[0].pie([counts[0], counts[1]],
            labels=['Legitimate\n(96.5%)', 'Fraud\n(3.5%)'],
            colors=['#3FB950', '#F78166'],
            autopct='%1.2f%%', startangle=90,
            textprops={'color': 'white', 'fontsize': 12})
axes[0].set_title('Class Distribution', color='white', fontsize=13)

bars = axes[1].bar(['Legitimate', 'Fraud'], [counts[0], counts[1]],
                   color=['#3FB950', '#F78166'], alpha=0.85)
for bar, count in zip(bars, [counts[0], counts[1]]):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height()*1.01,
                f'{count:,}', ha='center', va='bottom', color='white', fontsize=11)
axes[1].set_title('Transaction Counts', color='white', fontsize=13)
axes[1].set_ylabel('Count', color='white')

plt.suptitle('Class Imbalance: The Core Challenge', fontsize=14, color='white')
plt.tight_layout()
plt.savefig('/kaggle/working/outputs/plots/04_class_imbalance.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.close()

print("Class imbalance plot saved")

# STEP 6: Correlation Analysis
print("\n[6] Correlation Analysis...")

key_features = ['isFraud', 'TransactionAmt', 'card1', 'card2', 'card3',
                'card5', 'addr1', 'addr2', 'dist1', 'dist2',
                'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8',
                'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'hour']

corr_df = train[key_features].fillna(-999)
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True

sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', center=0,
            annot=False, fmt='.2f', square=True, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=14, color='white')
plt.tight_layout()
plt.savefig('/kaggle/working/outputs/plots/05_correlation_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.close()

fraud_corr = corr_matrix['isFraud'].drop('isFraud').sort_values(key=abs, ascending=False)
print("\nTop 10 features correlated with isFraud:")
print(fraud_corr.head(10))

print("\nPHASE 1 COMPLETE")


[1.1] Loading data...
Transaction data: (590540, 394)
Identity data: (144233, 41)
After merge: (590540, 434)
Memory usage: 2.64 GB

[1.2] Dataset overview...

Shape: (590540, 434)

Data types:
float64    399
object      31
int64        4
Name: count, dtype: int64

=== CLASS DISTRIBUTION ===
Total transactions: 590,540
Fraudulent: 20,663 (3.50%)
Legitimate: 569,877 (96.50%)

[1.3] Missing value analysis...
Features with >50% missing: 214
Complete features (0% missing): 20
Missing values plot saved

[2] Fraud vs Non-Fraud Analysis...

=== TRANSACTION AMOUNT STATISTICS ===

Fraud transactions:
count    20663.000000
mean       149.244779
std        232.212163
min          0.292000
25%         35.044000
50%         75.000000
75%        161.000000
max       5191.000000
Name: TransactionAmt, dtype: float64

Legit transactions:
count    569877.000000
mean        134.511665
std         239.395078
min           0.251000
25%          43.970000
50%          68.500000
75%         120.000000
max   

## PHASE 2: FEATURE ENGINEERING & PREPROCESSING

In [3]:
print("\n[1] Creating features...")

train['log_amount'] = np.log1p(train['TransactionAmt'])

# Email domain: is it free?
free_domains = ['gmail.com', 'yahoo.com', 'hotmail.com', 'gmail', 'yahoo', 'hotmail', 'outlook']
train['is_free_email'] = train['P_emaildomain'].isin(free_domains).astype(int)

# Select features
CATEGORICAL = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M4']
NUMERIC_BASE = ['TransactionAmt', 'log_amount', 'hour', 'day_of_week',
                'card1', 'card2', 'card3', 'card5',
                'addr1', 'addr2', 'dist1', 'dist2', 'is_free_email']
C_COLS = [f'C{i}' for i in range(1, 15)]
D_COLS = [f'D{i}' for i in range(1, 16)]
V_COLS = [f'V{i}' for i in [1,2,3,4,5,12,13,14,15,17,18,19,20,45,46,48,49,50,51]]

ALL_FEATURES = NUMERIC_BASE + C_COLS + D_COLS + V_COLS + CATEGORICAL

print(f"Total features: {len(ALL_FEATURES)}")

# Encode categoricals
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in CATEGORICAL:
    if col in train.columns:
        le = LabelEncoder()
        train[col] = train[col].fillna('Unknown')
        train[col] = le.fit_transform(train[col].astype(str))
        label_encoders[col] = le

# Prepare X and y
X = train[[f for f in ALL_FEATURES if f in train.columns]].copy()
y = train['isFraud'].values

# Handle missing values
v_in_X = [c for c in V_COLS if c in X.columns]
X[v_in_X] = X[v_in_X].fillna(-999)
X = X.fillna(X.median())

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Train/val split
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nX_train: {X_train.shape}, Fraud rate: {y_train.mean():.4f}")
print(f"X_val: {X_val.shape}, Fraud rate: {y_val.mean():.4f}")

print("\nPHASE 2 COMPLETE")


[1] Creating features...
Total features: 67
X shape: (590540, 67)
y shape: (590540,)

X_train: (472432, 67), Fraud rate: 0.0350
X_val: (118108, 67), Fraud rate: 0.0350

PHASE 2 COMPLETE


## PHASE 3: LOGISTIC REGRESSION FROM SCRATCH

In [4]:
print("\n[1] Implementing Logistic Regression with NumPy...")

class LogisticRegressionFromScratch:
    '''Logistic Regression using only NumPy'''
    
    def __init__(self, learning_rate=0.01, n_iterations=1000, lambda_reg=0.01):
        self.lr = learning_rate
        self.n_iter = n_iterations
        self.lambda_reg = lambda_reg
        self.weights = None
        self.bias = None
        self.loss_history = []
    
    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    def fit(self, X, y):
        m, n = X.shape
        self.weights = np.zeros(n)
        self.bias = 0.0
        
        for i in range(self.n_iter):
            # Forward pass
            z = np.dot(X, self.weights) + self.bias
            predictions = self._sigmoid(z)
            
            # Loss with L2 regularization
            epsilon = 1e-15
            p = np.clip(predictions, epsilon, 1 - epsilon)
            cross_entropy = -np.mean(y * np.log(p) + (1-y) * np.log(1-p))
            l2_penalty = (self.lambda_reg / (2*m)) * np.sum(self.weights**2)
            loss = cross_entropy + l2_penalty
            self.loss_history.append(loss)
            
            # Backpropagation
            error = predictions - y
            dw = (1/m) * np.dot(X.T, error) + (self.lambda_reg/m) * self.weights
            db = (1/m) * np.sum(error)
            
            # Update
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
            if i % 100 == 0:
                print(f'Iteration {i:4d}/{self.n_iter} | Loss: {loss:.6f}')
        
        return self
    
    def predict_proba(self, X):
        z = np.dot(X, self.weights) + self.bias
        return self._sigmoid(z)
    
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

# Scale features for logistic regression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("\n[2] Training Logistic Regression...")

model_lr = LogisticRegressionFromScratch(
    learning_rate=0.1,
    n_iterations=500,
    lambda_reg=0.01
)
model_lr.fit(X_train_scaled, y_train)

# Plot loss curve
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(model_lr.loss_history, color='#58A6FF', linewidth=2)
ax.set_title('Training Loss Over Iterations', color='white')
ax.set_xlabel('Iteration')
ax.set_ylabel('Binary Cross-Entropy Loss')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/outputs/plots/06_loss_curve_lr.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.close()

print("Loss curve saved")

# Evaluate
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, roc_curve

y_pred_proba_lr = model_lr.predict_proba(X_val_scaled)
y_pred_binary_lr = model_lr.predict(X_val_scaled, threshold=0.5)

auc_lr = roc_auc_score(y_val, y_pred_proba_lr)
print(f'\n=== LOGISTIC REGRESSION EVALUATION ===')
print(f'AUC-ROC: {auc_lr:.4f}')

print('\nClassification Report:')
print(classification_report(y_val, y_pred_binary_lr, target_names=['Legit', 'Fraud']))

cm = confusion_matrix(y_val, y_pred_binary_lr)
print('\nConfusion Matrix:')
print(f'True Negatives (TN): {cm[0][0]:,}')
print(f'False Positives (FP): {cm[0][1]:,}')
print(f'False Negatives (FN): {cm[1][0]:,}')
print(f'True Positives (TP): {cm[1][1]:,}')

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_val, y_pred_proba_lr)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='#58A6FF', linewidth=2, label=f'Logistic Regression (AUC={auc_lr:.3f})')
ax.plot([0,1],[0,1], 'k--', alpha=0.5, label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate (FPR)')
ax.set_ylabel('True Positive Rate (TPR / Recall)')
ax.set_title('ROC Curve - Logistic Regression', color='white')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/outputs/plots/07_roc_curve_lr.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.close()

print("ROC curve saved")

# Save logistic regression model
import pickle

with open('/kaggle/working/models/logistic_model.pkl', 'wb') as f:
    pickle.dump(model_lr, f)
with open('/kaggle/working/models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("\nPHASE 3 COMPLETE")


[1] Implementing Logistic Regression with NumPy...

[2] Training Logistic Regression...
Iteration    0/500 | Loss: 0.693147
Iteration  100/500 | Loss: 0.190839
Iteration  200/500 | Loss: 0.153483
Iteration  300/500 | Loss: 0.142877
Iteration  400/500 | Loss: 0.138318
Loss curve saved

=== LOGISTIC REGRESSION EVALUATION ===
AUC-ROC: 0.7844

Classification Report:
              precision    recall  f1-score   support

       Legit       0.97      1.00      0.98    113975
       Fraud       0.00      0.00      0.00      4133

    accuracy                           0.97    118108
   macro avg       0.48      0.50      0.49    118108
weighted avg       0.93      0.97      0.95    118108


Confusion Matrix:
True Negatives (TN): 113,975
False Positives (FP): 0
False Negatives (FN): 4,133
True Positives (TP): 0
ROC curve saved

PHASE 3 COMPLETE


## PHASE 4: XGBOOST WITH SMOTE

In [5]:
print("\n[1] Applying SMOTE to balance training data...")

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

print('Class distribution BEFORE SMOTE:')
print(f' Legit: {(y_train==0).sum():,}')
print(f' Fraud: {(y_train==1).sum():,}')

smote = SMOTE(
    sampling_strategy=0.5,
    random_state=42,
    k_neighbors=5
)

X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print('\nClass distribution AFTER SMOTE:')
print(f' Legit: {(y_train_balanced==0).sum():,}')
print(f' Fraud: {(y_train_balanced==1).sum():,}')
print(f' New fraud rate: {y_train_balanced.mean():.4f}')

print("\n[2] Training XGBoost...")

model_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    scale_pos_weight=1,
    eval_metric='auc',
    use_label_encoder=False,
    random_state=42,
    early_stopping_rounds=30,
    verbosity=0  # Set to 1 to see progress
)

model_xgb.fit(
    X_train_balanced, y_train_balanced,
    eval_set=[(X_val, y_val)],
    verbose=0
)

print(f'Best iteration: {model_xgb.best_iteration}')

# Evaluate
y_pred_proba_xgb = model_xgb.predict_proba(X_val)[:, 1]
auc_xgb = roc_auc_score(y_val, y_pred_proba_xgb)
y_pred_binary_xgb = (y_pred_proba_xgb >= 0.5).astype(int)

print(f'\n=== XGBOOST EVALUATION ===')
print(f'XGBoost AUC-ROC: {auc_xgb:.4f}')
print(f'Logistic Regression AUC: {auc_lr:.4f}')
print(f'Improvement: {(auc_xgb - auc_lr)*100:.2f}%')

print('\nClassification Report:')
print(classification_report(y_val, y_pred_binary_xgb, target_names=['Legit', 'Fraud']))

# Feature importance
feat_imp = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model_xgb.feature_importances_
}).sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(feat_imp['feature'][::-1], feat_imp['importance'][::-1],
        color='#58A6FF', alpha=0.85)
ax.set_xlabel('Importance Score')
ax.set_title('XGBoost Feature Importance (Top 20)', color='white')
plt.tight_layout()
plt.savefig('/kaggle/working/outputs/plots/08_feature_importance.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.close()

print("\nFeature importance plot saved")
print('\nTop 10 most important features:')
print(feat_imp[['feature','importance']].head(10).to_string(index=False))

# ROC Curve comparison
fpr_xgb, tpr_xgb, _ = roc_curve(y_val, y_pred_proba_xgb)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(fpr, tpr, color='#58A6FF', linewidth=2, label=f'Logistic Regression (AUC={auc_lr:.3f})')
ax.plot(fpr_xgb, tpr_xgb, color='#F78166', linewidth=2, label=f'XGBoost (AUC={auc_xgb:.3f})')
ax.plot([0,1],[0,1], 'k--', alpha=0.5, label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate (FPR)')
ax.set_ylabel('True Positive Rate (TPR / Recall)')
ax.set_title('ROC Curve Comparison', color='white')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/outputs/plots/09_roc_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.close()

print("ROC comparison plot saved")

# Save XGBoost model
model_xgb.save_model('/kaggle/working/models/xgb_model.json')

import joblib
joblib.dump(label_encoders, '/kaggle/working/models/label_encoders.pkl')

print("\nPHASE 4 COMPLETE")


[1] Applying SMOTE to balance training data...
Class distribution BEFORE SMOTE:
 Legit: 455,902
 Fraud: 16,530

Class distribution AFTER SMOTE:
 Legit: 455,902
 Fraud: 227,951
 New fraud rate: 0.3333

[2] Training XGBoost...
Best iteration: 299

=== XGBOOST EVALUATION ===
XGBoost AUC-ROC: 0.9042
Logistic Regression AUC: 0.7844
Improvement: 11.98%

Classification Report:
              precision    recall  f1-score   support

       Legit       0.98      0.99      0.99    113975
       Fraud       0.66      0.50      0.57      4133

    accuracy                           0.97    118108
   macro avg       0.82      0.75      0.78    118108
weighted avg       0.97      0.97      0.97    118108


Feature importance plot saved

Top 10 most important features:
feature  importance
     C8    0.141581
     C4    0.082991
     D3    0.042948
    V48    0.040317
     M4    0.039976
    C14    0.038430
     D8    0.036759
  card6    0.035074
    C13    0.027532
    C11    0.026878
ROC comparison 

## PHASE 5: GEOSPATIAL FRAUD ANALYSIS

In [6]:
print("\n[1] Creating synthetic coordinates for mapping...")

# Since IEEE dataset doesn't have true lat/lon, create synthetic ones for demo
np.random.seed(42)

# Get fraud transactions
fraud_tx = train[train['isFraud'] == 1].copy()

# Create synthetic US coordinates (demo purposes)
n = len(fraud_tx)
fraud_tx['lat'] = np.random.uniform(25, 49, n)
fraud_tx['lon'] = np.random.uniform(-125, -66, n)

# Weight by addr1 to create clusters
fraud_tx['lat'] += (fraud_tx['addr1'].fillna(0) % 10) * 0.5
fraud_tx['lon'] -= (fraud_tx['addr1'].fillna(0) % 15) * 0.3
fraud_tx['lat'] = fraud_tx['lat'].clip(24, 50)
fraud_tx['lon'] = fraud_tx['lon'].clip(-126, -65)

print(f"Fraud transactions for mapping: {len(fraud_tx):,}")

# Try to import folium, if not available, create alternative visualization
try:
    import folium
    from folium.plugins import HeatMap
    
    print("\n[2] Creating Folium heatmap...")
    
    m = folium.Map(
        location=[39.5, -98.35],  # Center of US
        zoom_start=4,
        tiles='CartoDB dark_matter',
        prefer_canvas=True
    )
    
    # Prepare heatmap data
    heat_data = fraud_tx[['lat', 'lon', 'TransactionAmt']].copy()
    heat_data['weight'] = np.log1p(heat_data['TransactionAmt'])
    heat_list = heat_data[['lat', 'lon', 'weight']].values.tolist()
    
    # Add heatmap
    HeatMap(
        heat_list,
        min_opacity=0.2,
        max_opacity=0.8,
        radius=15,
        blur=10,
        gradient={
            0.2: 'blue',
            0.4: 'cyan',
            0.6: 'lime',
            0.8: 'yellow',
            1.0: 'red'
        }
    ).add_to(m)
    
    # Save
    m.save('/kaggle/working/outputs/fraud_heatmap.html')
    print("Fraud heatmap saved (open in browser)")
    
except ImportError:
    print("Note: Folium not available, creating alternative visualization...")
    # Create scatter plot instead
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.scatter(fraud_tx['lon'], fraud_tx['lat'], 
               c=np.log1p(fraud_tx['TransactionAmt']), 
               cmap='YlOrRd', s=30, alpha=0.6)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('Fraud Hotspots (Synthetic Coordinates)', color='white')
    plt.colorbar(ax.collections[0], label='Log(Transaction Amount)')
    plt.tight_layout()
    plt.savefig('/kaggle/working/outputs/plots/10_fraud_hotspots.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
    plt.close()
    print("Fraud hotspots plot saved")

# Haversine distance function (for reference)
def haversine_distance(lat1, lon1, lat2, lon2):
    '''Calculate distance in km between two GPS points'''
    R = 6371  # Earth's radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

# Example
dist_km = haversine_distance(40.7, -74.0, 34.1, -118.2)
print(f"\nHaversine Distance Example:")
print(f"NYC to LA: {dist_km:.0f} km")
print(f"At 900 km/h (plane speed): {dist_km/900:.1f} hours minimum")

print("\nPHASE 5 COMPLETE")


[1] Creating synthetic coordinates for mapping...
Fraud transactions for mapping: 20,663

[2] Creating Folium heatmap...
Fraud heatmap saved (open in browser)

Haversine Distance Example:
NYC to LA: 3931 km
At 900 km/h (plane speed): 4.4 hours minimum

PHASE 5 COMPLETE


## PHASE 6: FASTAPI & MODEL SERVING

In [7]:
print("\n[1] Creating FastAPI application code...")

fastapi_code = '''
# api/main.py - FastAPI Application for Fraud Detection

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import numpy as np
import pickle
import json
from xgboost import XGBClassifier

# Initialize app
app = FastAPI(
    title='Fraud Detection API',
    description='Real-time credit card fraud detection using XGBoost',
    version='1.0.0'
)

# Load models at startup
model = XGBClassifier()
model.load_model('models/xgb_model.json')

with open('models/label_encoders.pkl', 'rb') as f:
    label_encoders = pickle.load(f)

# Define input schema
class Transaction(BaseModel):
    TransactionAmt: float = Field(..., example=125.50, description='Amount in USD')
    ProductCD: str = Field(..., example='W', description='Product code')
    card4: str = Field(..., example='visa', description='Card network')
    hour: int = Field(..., example=14, description='Hour of day 0-23')
    C1: float = Field(default=1.0, description='Count feature')
    C2: float = Field(default=1.0, description='Count feature')
    addr1: float = Field(default=299.0, description='Billing address code')

# Define output schema
class PredictionResponse(BaseModel):
    fraud_probability: float
    is_fraud: bool
    risk_level: str
    threshold_used: float

# Health check
@app.get('/')
def root():
    return {'status': 'ok', 'model': 'XGBoost Fraud Detector v1.0'}

@app.get('/health')
def health_check():
    return {'status': 'healthy', 'model_loaded': model is not None}

# Main prediction endpoint
@app.post('/predict', response_model=PredictionResponse)
def predict(transaction: Transaction):
    try:
        # Build feature array
        features = np.array([[
            transaction.TransactionAmt,
            np.log1p(transaction.TransactionAmt),
            transaction.hour,
            transaction.C1,
            transaction.C2,
            transaction.addr1,
        ]])
        
        # Get fraud probability
        prob = float(model.predict_proba(features)[0, 1])
        
        # Determine risk level
        if prob < 0.3:
            risk = 'LOW'
        elif prob < 0.7:
            risk = 'MEDIUM'
        else:
            risk = 'HIGH'
        
        return PredictionResponse(
            fraud_probability=prob,
            is_fraud=prob >= 0.5,
            risk_level=risk,
            threshold_used=0.5
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# Run with: uvicorn main:app --reload --host 0.0.0.0 --port 8000
'''

# Save FastAPI code
with open('/kaggle/working/outputs/fastapi_main.py', 'w') as f:
    f.write(fastapi_code)

print("FastAPI code saved to: /kaggle/working/outputs/fastapi_main.py")

print("\nPHASE 6 COMPLETE")


[1] Creating FastAPI application code...
FastAPI code saved to: /kaggle/working/outputs/fastapi_main.py

PHASE 6 COMPLETE


## SUMMARY & METRICS

In [8]:
print("\n[FINAL METRICS]")
print(f"Logistic Regression AUC-ROC: {auc_lr:.4f}")
print(f"XGBoost AUC-ROC: {auc_xgb:.4f}")
print(f"Improvement: {(auc_xgb - auc_lr)*100:.2f}%")

print("\n[MODEL FILES SAVED]")
print("/kaggle/working/models/xgb_model.json")
print("/kaggle/working/models/logistic_model.pkl")
print("/kaggle/working/models/scaler.pkl")
print("/kaggle/working/models/label_encoders.pkl")

print("\n[VISUALIZATIONS SAVED]")
import os
plot_files = os.listdir('/kaggle/working/outputs/plots')
for i, f in enumerate(sorted(plot_files), 1):
    print(f"  {i}. {f}")



[FINAL METRICS]
Logistic Regression AUC-ROC: 0.7844
XGBoost AUC-ROC: 0.9042
Improvement: 11.98%

[MODEL FILES SAVED]
/kaggle/working/models/xgb_model.json
/kaggle/working/models/logistic_model.pkl
/kaggle/working/models/scaler.pkl
/kaggle/working/models/label_encoders.pkl

[VISUALIZATIONS SAVED]
  1. 01_missing_values_dist.png
  2. 02_amount_distribution.png
  3. 03_time_patterns.png
  4. 04_class_imbalance.png
  5. 05_correlation_heatmap.png
  6. 06_loss_curve_lr.png
  7. 07_roc_curve_lr.png
  8. 08_feature_importance.png
  9. 09_roc_comparison.png
